# <font color="#418FDE" size="6.5" uppercase>**Cross Decomposition**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply PLS and CCA methods to paired feature-target data. 
- Interpret latent scores, loadings, and predictions from cross-decomposition models. 
- Select component counts and scaling choices for cross-decomposition workflows. 


## **1. PLS Methods**

### **1.1. PLS Objectives**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_01.jpg?v=1783966980" width="250">



>* Find linked patterns between features and targets
>* Reduce noisy data into relevant latent components

>* Feature variance alone may miss target relevance
>* PLS finds supervised cross-block components

>* PLS links complex data to meaningful outcomes
>* Latent components reveal coordinated variable patterns



### **1.2. PLS Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_02.jpg?v=1783966982" width="250">



>* Predict targets from noisy, correlated features
>* Extract target-relevant latent components

>* Build target-relevant latent scores from paired data
>* Predict new cases using compact feature summaries

>* Predicts related targets using shared components
>* Components emphasize prediction and interpretable links



In [ ]:
#@title Python Code - PLS Regression

# Demonstrate PLS regression without external machine learning.
# Latent scores connect features with targets.
# Small synthetic data keeps output readable.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducible results.
rng = np.random.default_rng(42)

# Create paired feature and target blocks.
n_samples, n_features, n_targets = 40, 6, 2
latent = rng.normal(size=(n_samples, 2))

# Build correlated features from hidden factors.
feature_weights = rng.normal(size=(2, n_features))
X = latent @ feature_weights + rng.normal(scale=0.25, size=(n_samples, n_features))

# Build two related target measurements.
target_weights = np.array([[2.0, -1.0], [0.5, 1.5]])
Y = latent @ target_weights + rng.normal(scale=0.20, size=(n_samples, n_targets))

# Validate paired observations before modeling.
assert X.shape[0] == Y.shape[0]
assert X.shape[1] >= 2 and Y.shape[1] >= 1

# Split data into training and testing parts.
train_size = 30
X_train, X_test = X[:train_size], X[train_size:]
Y_train, Y_test = Y[:train_size], Y[train_size:]

# Standardize using training statistics only.
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)
Y_mean, Y_std = Y_train.mean(axis=0), Y_train.std(axis=0)

# Avoid division by zero during scaling.
X_std = np.where(X_std == 0, 1.0, X_std)
Y_std = np.where(Y_std == 0, 1.0, Y_std)

# Apply the same scaling to both splits.
Xs = (X_train - X_mean) / X_std
Ys = (Y_train - Y_mean) / Y_std
Xts = (X_test - X_mean) / X_std

# Choose a small number of PLS components.
n_components = 2
W, P, Q, T_scores = [], [], [], []

# Copy matrices for sequential deflation.
X_res = Xs.copy()
Y_res = Ys.copy()

# Extract components using a simple NIPALS loop.
for component in range(n_components):
    u = Y_res[:, [0]].copy()
    for iteration in range(20):
        w = X_res.T @ u
        w = w / max(np.linalg.norm(w), 1e-12)
        t = X_res @ w
        t_norm_sq = max(float(np.sum(t * t)), 1e-12)
        q = Y_res.T @ t / t_norm_sq
        q_norm_sq = max(float(np.sum(q * q)), 1e-12)
        u = Y_res @ q / q_norm_sq
    t_norm_sq = max(float(np.sum(t * t)), 1e-12)
    p = X_res.T @ t / t_norm_sq
    X_res = X_res - t @ p.T
    Y_res = Y_res - t @ q.T
    W.append(w.ravel())
    P.append(p.ravel())
    Q.append(q.ravel())
    T_scores.append(t.ravel())

# Convert collected component quantities into matrices.
W = np.column_stack(W)
P = np.column_stack(P)
Q = np.column_stack(Q)
T_scores = np.column_stack(T_scores)

# Compute regression coefficients in scaled space.
middle = np.linalg.inv(P.T @ W)
B_scaled = W @ middle @ Q.T

# Predict test targets and return original units.
Y_pred_scaled = Xts @ B_scaled
Y_pred = Y_pred_scaled * Y_std + Y_mean

# Measure prediction quality on held-out observations.
rmse = np.sqrt(np.mean((Y_test - Y_pred) ** 2))
score_corr = np.corrcoef(T_scores[:, 0], Ys[:, 0])[0, 1]

# Print compact teaching results only.
print(f"PLS components used: {n_components}")
print(f"Test RMSE across targets: {rmse:.3f}")
print(f"First score-target correlation: {score_corr:.3f}")
print(f"First feature loading sample: {P[:3, 0].round(3)}")

# Plot actual versus predicted first target.
plt.figure(figsize=(5, 4))
plt.scatter(Y_test[:, 0], Y_pred[:, 0], color="teal")
plt.xlabel("Actual target one")
plt.ylabel("Predicted target one")
plt.title("PLS regression prediction check")
plt.grid(True, alpha=0.3)
plt.show()



### **1.3. PLS Canonical**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_03.jpg?v=1783966985" width="250">



>* Find shared patterns between paired data blocks
>* Compress correlated measurements into linked components

>* Build paired components with strong shared covariance
>* Reveal coordinated patterns across many variables

>* PLS emphasizes shared covariance over correlation
>* Scores and loadings reveal linked patterns



In [ ]:
#@title Python Code - PLS Canonical

# Demonstrate PLS Canonical without external machine learning.
# Two paired blocks share hidden variation.
# Scores reveal linked multivariate patterns.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic paired measurement blocks.
rng = np.random.default_rng(7)
n_samples, n_components = 80, 2

# Build shared latent environmental patterns.
shared = rng.normal(size=(n_samples, n_components))
noise_x = rng.normal(scale=0.35, size=(n_samples, 4))
noise_y = rng.normal(scale=0.35, size=(n_samples, 3))

# Mix latent patterns into two blocks.
wx_true = np.array([[0.9, 0.1], [0.7, -0.2], [-0.4, 0.8], [0.1, 0.9]])
wy_true = np.array([[0.8, 0.2], [-0.5, 0.7], [0.2, 0.9]])
X = shared @ wx_true.T + noise_x
Y = shared @ wy_true.T + noise_y

# Standardize each block before decomposition.
X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0, ddof=1)

# Validate paired rows before cross decomposition.
assert X.shape[0] == Y.shape[0]
assert X.shape[1] >= n_components
assert Y.shape[1] >= n_components

# Use covariance singular vectors as PLS weights.
cross_cov = X.T @ Y / (n_samples - 1)
U, singular_values, Vt = np.linalg.svd(cross_cov, full_matrices=False)

# Compute paired latent scores for both blocks.
X_scores = X @ U[:, :n_components]
Y_scores = Y @ Vt.T[:, :n_components]
score_corr = np.corrcoef(X_scores[:, 0], Y_scores[:, 0])[0, 1]

# Estimate simple loadings for interpretation.
x_loadings = X.T @ X_scores / (n_samples - 1)
y_loadings = Y.T @ Y_scores / (n_samples - 1)

# Print compact teaching summary.
print("PLS Canonical via covariance SVD")
print(f"Block shapes: X={X.shape}, Y={Y.shape}")
print(f"First shared covariance strength: {singular_values[0]:.3f}")
print(f"First score correlation: {score_corr:.3f}")
print("Top X loading index:", int(np.argmax(abs(x_loadings[:, 0]))))
print("Top Y loading index:", int(np.argmax(abs(y_loadings[:, 0]))))

# Plot paired first-component scores.
plt.figure(figsize=(6, 4))
plt.scatter(X_scores[:, 0], Y_scores[:, 0], alpha=0.75)
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.xlabel("X block latent score 1")
plt.ylabel("Y block latent score 1")
plt.title("PLS Canonical paired scores")
plt.tight_layout()
plt.show()



## **2. CCA Latent Interpretation**

### **2.1. SVD PLS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_01.jpg?v=1783966974" width="250">



>* Find paired directions that vary together
>* Interpret scores as shared data patterns

>* Components reflect weighted patterns across variables
>* Interpret signs using context, plots, and scaling

>* Link predictions to latent scores and loadings
>* Validate components, scaling, and training similarity



In [ ]:
#@title Python Code - SVD PLS

# Demonstrate SVD PLS latent interpretation.
# Use tiny paired synthetic measurements.
# Avoid external machine learning libraries.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic paired feature and target blocks.
rng = np.random.default_rng(7)
n_samples = 36
latent = rng.normal(size=(n_samples, 2))

# Build features from shared hidden patterns.
X_weights_true = np.array([[0.9, 0.1], [0.7, -0.2], [-0.4, 0.8], [0.1, 0.9]])
Y_weights_true = np.array([[0.8, -0.1], [-0.5, 0.7]])
X = latent @ X_weights_true.T + rng.normal(scale=0.18, size=(n_samples, 4))

# Build targets from related hidden patterns.
Y = latent @ Y_weights_true.T + rng.normal(scale=0.18, size=(n_samples, 2))
assert X.shape[0] == Y.shape[0]
assert X.shape[1] == 4 and Y.shape[1] == 2

# Standardize both blocks before decomposition.
X_mean, X_std = X.mean(axis=0), X.std(axis=0, ddof=1)
Y_mean, Y_std = Y.mean(axis=0), Y.std(axis=0, ddof=1)
Xs = (X - X_mean) / X_std
Ys = (Y - Y_mean) / Y_std

# Decompose cross covariance using singular value decomposition.
C = Xs.T @ Ys / (n_samples - 1)
U, singular_values, Vt = np.linalg.svd(C, full_matrices=False)
x_scores = Xs @ U[:, :2]
y_scores = Ys @ Vt.T[:, :2]

# Fit a small latent regression for predictions.
coef = np.linalg.lstsq(x_scores, Ys, rcond=None)[0]
Y_pred_scaled = x_scores @ coef
rmse = np.sqrt(np.mean((Ys - Y_pred_scaled) ** 2))

# Summarize component strength and interpretation.
print(f"SVD PLS components: {len(singular_values)}")
print(f"First singular value: {singular_values[0]:.3f}")
print(f"Second singular value: {singular_values[1]:.3f}")
print(f"Scaled target RMSE: {rmse:.3f}")
print("Largest X loading on component 1: feature", int(np.argmax(np.abs(U[:, 0])) + 1))
print("Largest Y loading on component 1: target", int(np.argmax(np.abs(Vt.T[:, 0])) + 1))

# Plot paired latent scores for interpretation.
plt.figure(figsize=(6, 4))
plt.scatter(x_scores[:, 0], y_scores[:, 0], color="teal", alpha=0.8)
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.xlabel("Feature block score, component 1")
plt.ylabel("Target block score, component 1")
plt.title("SVD PLS: paired latent scores")
plt.tight_layout()
plt.show()



### **2.2. CCA Model Fitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_02.jpg?v=1783966976" width="250">



>* CCA finds aligned latent scores across datasets
>* High correlations reveal shared variation patterns

>* Weights form scores; loadings explain variables
>* Interpret broad patterns, not isolated coefficients

>* Scale data and interpret components cautiously
>* Validate latent patterns before drawing conclusions



In [ ]:
#@title Python Code - CCA Model Fitting

# This example fits CCA without sklearn.
# Synthetic paired tables share hidden patterns.
# Scores, loadings, and correlations are interpreted.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic paired observations.
rng = np.random.default_rng(7)
n_samples, x_features, y_features = 80, 4, 3

# Build two shared latent factors.
shared_one = rng.normal(size=n_samples)
shared_two = rng.normal(size=n_samples)

# Combine factors into the first table.
X = np.column_stack([
    0.9 * shared_one + 0.2 * rng.normal(size=n_samples),
    0.7 * shared_one + 0.3 * shared_two + 0.2 * rng.normal(size=n_samples),
    0.8 * shared_two + 0.2 * rng.normal(size=n_samples),
    rng.normal(size=n_samples),
])

# Combine factors into the second table.
Y = np.column_stack([
    0.8 * shared_one + 0.2 * rng.normal(size=n_samples),
    0.6 * shared_two + 0.2 * rng.normal(size=n_samples),
    0.5 * shared_one - 0.4 * shared_two + 0.2 * rng.normal(size=n_samples),
])

# Validate paired table dimensions.
assert X.shape[0] == Y.shape[0]
assert X.shape[1] >= 2 and Y.shape[1] >= 2

# Standardize columns before CCA fitting.
Xz = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
Yz = (Y - Y.mean(axis=0)) / Y.std(axis=0, ddof=1)

# Compute covariance blocks for CCA.
Sxx = Xz.T @ Xz / (n_samples - 1)
Syy = Yz.T @ Yz / (n_samples - 1)
Sxy = Xz.T @ Yz / (n_samples - 1)

# Add tiny ridge values for stability.
ridge = 1e-8
Sxx += ridge * np.eye(x_features)
Syy += ridge * np.eye(y_features)

# Whiten both covariance blocks.
ex, Ux = np.linalg.eigh(Sxx)
ey, Uy = np.linalg.eigh(Syy)
Wx = Ux @ np.diag(1 / np.sqrt(ex)) @ Ux.T
Wy = Uy @ np.diag(1 / np.sqrt(ey)) @ Uy.T

# Fit CCA through singular value decomposition.
M = Wx @ Sxy @ Wy
left, corr_values, right_t = np.linalg.svd(M, full_matrices=False)

# Convert singular vectors into canonical weights.
x_weights = Wx @ left[:, :2]
y_weights = Wy @ right_t.T[:, :2]

# Project observations into latent score spaces.
x_scores = Xz @ x_weights
y_scores = Yz @ y_weights

# Compute loadings as variable score correlations.
x_loadings = np.corrcoef(Xz.T, x_scores.T)[:x_features, x_features:]
y_loadings = np.corrcoef(Yz.T, y_scores.T)[:y_features, y_features:]

# Summarize fitted canonical relationships.
print(f"CCA components fitted: {x_scores.shape[1]}")
print(f"First canonical correlation: {corr_values[0]:.3f}")
print(f"Second canonical correlation: {corr_values[1]:.3f}")
print("Top X loading on component 1: X" + str(np.argmax(abs(x_loadings[:, 0])) + 1))
print("Top Y loading on component 1: Y" + str(np.argmax(abs(y_loadings[:, 0])) + 1))
print("Scores place each observation in paired latent spaces.")

# Plot paired first-component scores.
plt.figure(figsize=(6, 4))
plt.scatter(x_scores[:, 0], y_scores[:, 0], alpha=0.75)
plt.xlabel("X canonical score 1")
plt.ylabel("Y canonical score 1")
plt.title("CCA fitted latent scores")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Covariance Objectives**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_03.jpg?v=1783966978" width="250">



>* Find paired summaries that move together
>* Latent scores reveal shared data structure

>* Focus on shared variation, not size
>* Components reveal meaningful joint behavior patterns

>* Predictions project through shared latent patterns
>* Validate patterns against noise and domain knowledge



## **3. Component Selection Practice**

### **3.1. Scaling Paired Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_01.jpg?v=1783966987" width="250">



>* Scale paired variables for fair comparison
>* Prevent units from distorting shared patterns

>* Center variables; scale when units differ
>* Preserve meaningful variance when scale matters

>* Fit scaling only on training data
>* Validate scaling choices with component selection



In [ ]:
#@title Python Code - Scaling Paired Data

# Demonstrate scaling for paired data.
# Compare raw and standardized blocks.
# Use tiny deterministic synthetic measurements.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible paired observations.
rng = np.random.default_rng(7)
n_samples = 40
latent = rng.normal(size=n_samples)

# Build predictor block with mixed units.
height_cm = 170 + 8 * latent + rng.normal(size=n_samples)
steps = 6000 + 900 * latent + rng.normal(scale=300, size=n_samples)
score = 3 + 0.4 * latent + rng.normal(scale=0.2, size=n_samples)

# Build target block with different scales.
blood_pressure = 120 + 5 * latent + rng.normal(scale=2, size=n_samples)
cholesterol = 190 + 20 * latent + rng.normal(scale=8, size=n_samples)

# Stack paired blocks into matrices.
X = np.column_stack([height_cm, steps, score])
Y = np.column_stack([blood_pressure, cholesterol])

# Validate paired sample counts.
assert X.shape[0] == Y.shape[0]
assert X.shape[1] == 3 and Y.shape[1] == 2

# Define safe standardization helper.
def standardize(block):
    means = block.mean(axis=0)
    scales = block.std(axis=0, ddof=1)
    scales = np.where(scales == 0, 1, scales)
    return (block - means) / scales

# Compute raw and scaled cross-covariances.
raw_cross = X.T @ Y / (n_samples - 1)
scaled_cross = standardize(X).T @ standardize(Y) / (n_samples - 1)

# Summarize dominance before and after scaling.
raw_max = np.unravel_index(np.abs(raw_cross).argmax(), raw_cross.shape)
scaled_max = np.unravel_index(np.abs(scaled_cross).argmax(), scaled_cross.shape)

# Print compact teaching results.
print("Raw strongest pair index:", raw_max)
print("Scaled strongest pair index:", scaled_max)
print("Raw cross-covariance range:", round(np.ptp(raw_cross), 2))
print("Scaled cross-covariance range:", round(np.ptp(scaled_cross), 2))

# Plot comparable cross-block association matrices.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
labels_x = ["height", "steps", "score"]
labels_y = ["pressure", "cholesterol"]

# Draw raw matrix heatmap.
im0 = axes[0].imshow(raw_cross, cmap="coolwarm")
axes[0].set_title("Raw covariance")
axes[0].set_xticks(range(2), labels_y)
axes[0].set_yticks(range(3), labels_x)

# Draw scaled matrix heatmap.
im1 = axes[1].imshow(scaled_cross, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Scaled covariance")
axes[1].set_xticks(range(2), labels_y)
axes[1].set_yticks(range(3), labels_x)

# Add compact colorbars and layout.
fig.colorbar(im0, ax=axes[0], fraction=0.046)
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()



### **3.2. Choosing Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_02.jpg?v=1783966991" width="250">



>* Components capture shared patterns between paired data.
>* Choose enough components without fitting noise.

>* Validate component counts on unseen data
>* Prefer simpler, stable, interpretable models

>* Scaling affects which components seem useful
>* Choose stable, interpretable, predictive components



In [ ]:
#@title Python Code - Choosing Components

# Practice choosing cross decomposition components.
# Validation helps avoid noisy components.
# Scaling changes which patterns dominate.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible paired feature and target blocks.
rng = np.random.default_rng(7)
samples, features, targets = 80, 6, 2

# Build two shared latent signals.
latent = rng.normal(size=(samples, 2))
x_weights = rng.normal(size=(2, features))
y_weights = rng.normal(size=(2, targets))

# Add noise and one large scale feature.
X = latent @ x_weights + rng.normal(scale=0.35, size=(samples, features))
Y = latent @ y_weights + rng.normal(scale=0.35, size=(samples, targets))
X[:, 0] = X[:, 0] * 25.0

# Split data into training and validation sets.
train_size = 55
X_train, X_valid = X[:train_size], X[train_size:]
Y_train, Y_valid = Y[:train_size], Y[train_size:]

# Standardize arrays using training statistics only.
def standardize(train, valid):
    mean = train.mean(axis=0, keepdims=True)
    scale = train.std(axis=0, keepdims=True)
    scale = np.where(scale == 0, 1.0, scale)
    return (train - mean) / scale, (valid - mean) / scale

# Fit a tiny PLS style model with NIPALS.
def fit_pls_predict(Xtr, Ytr, Xva, components):
    Xc = Xtr - Xtr.mean(axis=0, keepdims=True)
    Yc = Ytr - Ytr.mean(axis=0, keepdims=True)
    Xv = Xva - Xtr.mean(axis=0, keepdims=True)
    W, P, Q = [], [], []
    for _ in range(components):
        u = Yc[:, [0]]
        for _ in range(20):
            w = Xc.T @ u
            w = w / max(np.linalg.norm(w), 1e-12)
            t = Xc @ w
            t_norm_sq = max(float(np.sum(t * t)), 1e-12)
            q = Yc.T @ t / t_norm_sq
            q_norm_sq = max(float(np.sum(q * q)), 1e-12)
            u = Yc @ q / q_norm_sq
        t_norm_sq = max(float(np.sum(t * t)), 1e-12)
        p = Xc.T @ t / t_norm_sq
        Xc = Xc - t @ p.T
        Yc = Yc - t @ q.T
        W.append(w.ravel()); P.append(p.ravel()); Q.append(q.ravel())
    W, P, Q = np.array(W).T, np.array(P).T, np.array(Q).T
    coef = W @ np.linalg.pinv(P.T @ W) @ Q.T
    return Xv @ coef + Ytr.mean(axis=0, keepdims=True)

# Compare component counts with and without scaling.
results = []
for scaled in [False, True]:
    Xtr, Xva = (X_train, X_valid)
    Ytr, Yva = (Y_train, Y_valid)
    if scaled:
        Xtr, Xva = standardize(X_train, X_valid)
        Ytr, Yva = standardize(Y_train, Y_valid)
    for comps in range(1, 5):
        pred = fit_pls_predict(Xtr, Ytr, Xva, comps)
        rmse = np.sqrt(np.mean((Yva - pred) ** 2))
        results.append(("scaled" if scaled else "unscaled", comps, rmse))

# Print a compact validation summary.
print("Validation RMSE by scaling choice and components:")
for label, comps, rmse in results:
    print(f"{label:8s} components={comps}: RMSE={rmse:.3f}")

# Plot validation error for visual component selection.
plt.figure(figsize=(7, 4))
for label in ["unscaled", "scaled"]:
    xs = [c for s, c, r in results if s == label]
    ys = [r for s, c, r in results if s == label]
    plt.plot(xs, ys, marker="o", label=label)
plt.xlabel("Number of components")
plt.ylabel("Validation RMSE")
plt.title("Choose simpler component counts when validation is similar")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Interpreting Loadings**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_03.jpg?v=1783966989" width="250">



>* Loadings explain what latent components represent
>* Large related loadings reveal meaningful feature patterns

>* Standardize data before comparing feature loadings
>* Trust stable early components over noisy later ones

>* Compare loading directions and variable groupings
>* Check stability before trusting component interpretations



In [ ]:
#@title Python Code - Interpreting Loadings

# This example interprets loadings without external machine learning.
# Synthetic paired blocks mimic features and outcomes.
# Scaling choices change which variables appear important.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic paired data blocks.
rng = np.random.default_rng(7)
n_samples = 40
latent = rng.normal(size=n_samples)

# Build features with intentionally different scales.
calories = 2200 + 450 * latent + rng.normal(0, 120, n_samples)
vitamin_c = 70 + 12 * latent + rng.normal(0, 5, n_samples)
steps = 7000 + 900 * latent + rng.normal(0, 500, n_samples)

# Build outcomes sharing the same latent pattern.
wellness = 50 + 8 * latent + rng.normal(0, 3, n_samples)
fatigue = 30 - 6 * latent + rng.normal(0, 3, n_samples)

# Store names for readable loading summaries.
feature_names = ["calories", "vitamin_c", "steps"]
outcome_names = ["wellness", "fatigue"]

# Combine arrays into paired data blocks.
X = np.column_stack([calories, vitamin_c, steps])
Y = np.column_stack([wellness, fatigue])

# Validate paired observations before decomposition.
assert X.shape[0] == Y.shape[0]
assert X.shape[1] == len(feature_names)

# Define a small standardization helper.
def standardize(block):
    means = block.mean(axis=0)
    scales = block.std(axis=0, ddof=1)
    return (block - means) / scales

# Compute first cross-decomposition direction using SVD.
def first_component_loadings(X_block, Y_block):
    cross_cov = X_block.T @ Y_block / (X_block.shape[0] - 1)
    left, values, right_t = np.linalg.svd(cross_cov, full_matrices=False)
    return left[:, 0], right_t.T[:, 0], values[0]

# Compare raw and standardized loading interpretations.
x_raw, y_raw, strength_raw = first_component_loadings(X, Y)
x_scaled, y_scaled, strength_scaled = first_component_loadings(standardize(X), standardize(Y))

# Put scaled feature loadings into a compact table.
loading_table = pd.DataFrame({"feature": feature_names, "loading": x_scaled})
loading_table["absolute_loading"] = loading_table["loading"].abs()
loading_table = loading_table.sort_values("absolute_loading", ascending=False)

# Print concise teaching output only.
print("Raw strongest feature:", feature_names[int(np.argmax(np.abs(x_raw)))])
print("Scaled strongest feature:", loading_table.iloc[0]["feature"])
print("Outcome directions:", dict(zip(outcome_names, np.round(y_scaled, 2))))
print("Shared strength raw versus scaled:", round(strength_raw, 1), round(strength_scaled, 2))
print("Interpret signs relatively; components can be flipped.")

# Plot standardized feature loadings for interpretation.
plt.figure(figsize=(6, 3))
plt.bar(loading_table["feature"], loading_table["loading"], color="steelblue")
plt.axhline(0, color="black", linewidth=1)
plt.title("First Component Feature Loadings After Scaling")
plt.ylabel("Loading")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Cross Decomposition**</font>


In this lecture, you learned to:
- Apply PLS and CCA methods to paired feature-target data. 
- Interpret latent scores, loadings, and predictions from cross-decomposition models. 
- Select component counts and scaling choices for cross-decomposition workflows. 

In the next Module (Module 15), we will go over 'Bayes Calibration'